# 🏗️ Building ChatGPT: The Three Stages of Teaching an AI to Chat

Welcome to the most exciting chapter in GenAI! We're going to understand exactly how ChatGPT went from a dumb text-completion machine to the AI assistant that took over the world.

## What You'll Learn

By the end of this notebook, you'll understand:
- 🔄 **RoPE** -- How the model knows word ORDER (with a clock analogy!)
- 📚 **Stage 1: Pretraining** -- Reading the entire internet
- 🏫 **Stage 2: SFT** -- Learning to be a helpful assistant
- 📝 **Stage 3: RLHF** -- Learning what humans actually prefer
- 🎯 **DPO** -- A simpler alternative to full RLHF

**Prerequisites:** Chapters 02-03 (Transformers, attention, tokenization)

**Time:** ~60 minutes

---

## 🧭 The Big Picture: From Text Predictor to AI Assistant

```
┌──────────────────────────────────────────────────────────────────────────┐
│              THE CHATGPT TRAINING PIPELINE                              │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   RAW TRANSFORMER                                                       │
│   "I can predict the next word!"                                       │
│        │                                                                │
│        ▼                                                                │
│   ┌──────────────────────────────┐                                      │
│   │  STAGE 1: PRETRAINING       │  🎒 "Read the entire internet"       │
│   │  Trillions of tokens        │     Next-token prediction            │
│   │  Months of GPU training     │     Learn grammar, facts, reasoning  │
│   └──────────────┬───────────────┘                                      │
│                  │                                                      │
│                  ▼                                                      │
│   ┌──────────────────────────────┐                                      │
│   │  STAGE 2: SFT               │  🏫 "Learn to follow instructions"   │
│   │  ~100K demo examples        │     Supervised fine-tuning           │
│   │  Days of training           │     Learn helpful response format    │
│   └──────────────┬───────────────┘                                      │
│                  │                                                      │
│                  ▼                                                      │
│   ┌──────────────────────────────┐                                      │
│   │  STAGE 3: RLHF              │  📝 "Learn from human feedback"      │
│   │  ~300K preference pairs     │     Reward model + PPO              │
│   │  Iterative refinement       │     Optimize for human preferences  │
│   └──────────────┬───────────────┘                                      │
│                  │                                                      │
│                  ▼                                                      │
│   CHATGPT 🎉                                                           │
│   "I'm helpful, harmless, and honest!"                                 │
│                                                                         │
└──────────────────────────────────────────────────────────────────────────┘
```

Let's start with a crucial building block before diving into training...

---

## 🔄 RoPE: Rotary Positional Encoding

### ELI12: The Clock Analogy ⏰

Imagine you're reading a sentence: **"The cat sat on the mat"**

The model needs to know that "cat" is the 2nd word, "sat" is the 3rd, etc. But HOW do we encode position?

**The Clock Trick:**
- Imagine each word position has its own **clock**
- Position 1 rotates the clock hand by 1 tick
- Position 2 rotates by 2 ticks
- Position 5 rotates by 5 ticks

Now here's the magic: when the model compares two words, it looks at the **difference in their rotations**. Words 2 and 5 are always 3 ticks apart, no matter where they appear in the sentence!

This means RoPE naturally encodes **relative position** -- how far apart two words are -- which is what actually matters for understanding language.

### Staff-Level: The Math 🔢

RoPE applies a rotation matrix to the query and key vectors at each position:

```
For position m, the rotation applied to a 2D pair (x₁, x₂) is:

    ┌ cos(mθ)  -sin(mθ) ┐   ┌ x₁ ┐
    │                    │ × │     │
    └ sin(mθ)   cos(mθ) ┘   └ x₂ ┘

Where θ = 10000^(-2i/d) for the i-th dimension pair
```

**Key Property:** When computing attention `q_m^T · k_n`, the dot product depends only on `(m - n)` -- the relative distance! This is because:

```
R(mθ)^T · R(nθ) = R((m-n)θ)
```

Rotation matrices compose beautifully -- the product of two rotations is just the rotation of the difference!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Visualize RoPE: How rotations encode position
# ============================================================

def rope_rotation(position, dim_pair_idx, d_model=64, base=10000):
    """
    Compute the RoPE rotation angle for a given position and dimension pair.
    
    Args:
        position: Word position in sequence (0, 1, 2, ...)
        dim_pair_idx: Which dimension pair (0, 1, 2, ...)
        d_model: Model dimension
        base: Base for frequency computation (default 10000)
    
    Returns:
        Rotation angle in radians
    """
    theta = 1.0 / (base ** (2 * dim_pair_idx / d_model))
    return position * theta


def apply_rope_2d(x1, x2, angle):
    """
    Apply RoPE rotation to a 2D vector pair.
    
    This is the core of RoPE: multiply by a rotation matrix.
    
    ┌ cos(θ)  -sin(θ) ┐   ┌ x₁ ┐     ┌ x₁·cos(θ) - x₂·sin(θ) ┐
    │                  │ × │    │  =  │                          │
    └ sin(θ)   cos(θ) ┘   └ x₂ ┘     └ x₁·sin(θ) + x₂·cos(θ) ┘
    """
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    return x1 * cos_a - x2 * sin_a, x1 * sin_a + x2 * cos_a


# --- Visualization 1: RoPE rotations in 2D ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Show how a vector gets rotated at different positions
original_vec = np.array([1.0, 0.0])  # Start with unit vector along x-axis
positions = range(8)
colors = plt.cm.viridis(np.linspace(0, 1, len(positions)))

# Panel 1: Slow rotation (low frequency dimension pair)
ax1 = axes[0]
ax1.set_title('Dimension Pair 0\n(Slow Rotation -- Low Frequency)', fontsize=12, fontweight='bold')
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.3)
ax1.add_patch(circle)

for pos in positions:
    angle = rope_rotation(pos, dim_pair_idx=0, d_model=64)
    rx, ry = apply_rope_2d(original_vec[0], original_vec[1], angle)
    ax1.arrow(0, 0, rx*0.9, ry*0.9, head_width=0.06, head_length=0.04, 
              fc=colors[pos], ec=colors[pos], linewidth=2)
    ax1.annotate(f'pos={pos}', (rx*1.1, ry*1.1), fontsize=8, 
                 color=colors[pos], fontweight='bold')

# Panel 2: Fast rotation (high frequency dimension pair)
ax2 = axes[1]
ax2.set_title('Dimension Pair 15\n(Fast Rotation -- High Frequency)', fontsize=12, fontweight='bold')
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

circle2 = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.3)
ax2.add_patch(circle2)

for pos in positions:
    angle = rope_rotation(pos, dim_pair_idx=15, d_model=64)
    rx, ry = apply_rope_2d(original_vec[0], original_vec[1], angle)
    ax2.arrow(0, 0, rx*0.9, ry*0.9, head_width=0.06, head_length=0.04,
              fc=colors[pos], ec=colors[pos], linewidth=2)
    ax2.annotate(f'pos={pos}', (rx*1.1, ry*1.1), fontsize=8,
                 color=colors[pos], fontweight='bold')

# Panel 3: The key insight -- relative distance
ax3 = axes[2]
ax3.set_title('Key Insight: Relative Distance\n(Angle difference = position distance)', fontsize=12, fontweight='bold')
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)
ax3.set_aspect('equal')
ax3.grid(True, alpha=0.3)

circle3 = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.3)
ax3.add_patch(circle3)

# Show two pairs with same relative distance
pairs = [(1, 4, '#e74c3c', 'pos 1→4 (dist=3)'), (5, 8, '#3498db', 'pos 5→8 (dist=3)')]
for p1, p2, color, label in pairs:
    a1 = rope_rotation(p1, 0, d_model=64)
    a2 = rope_rotation(p2, 0, d_model=64)
    rx1, ry1 = apply_rope_2d(1.0, 0.0, a1)
    rx2, ry2 = apply_rope_2d(1.0, 0.0, a2)
    ax3.arrow(0, 0, rx1*0.9, ry1*0.9, head_width=0.05, head_length=0.03,
              fc=color, ec=color, linewidth=2, alpha=0.5)
    ax3.arrow(0, 0, rx2*0.9, ry2*0.9, head_width=0.05, head_length=0.03,
              fc=color, ec=color, linewidth=2)
    angle_diff = np.degrees(a2 - a1)
    ax3.annotate(f'{label}\nΔangle={angle_diff:.1f}°', 
                 (rx2*1.15, ry2*1.15), fontsize=8, color=color, fontweight='bold')

ax3.text(0, -1.35, 'Same distance = Same angle difference!\nThis is why RoPE encodes RELATIVE position.',
         ha='center', fontsize=9, style='italic', color='#666',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('🔄 RoPE: Rotary Positional Encoding Visualized', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY INSIGHT: RoPE encodes position as ROTATION")
print("="*70)
print("""
• Each position m rotates vectors by angle m·θ
• Different dimension pairs use different frequencies (θ values)
• Low-frequency pairs → capture long-range position info
• High-frequency pairs → capture fine-grained local position
• The dot product q_m · k_n depends only on (m-n) → RELATIVE position!

Why this matters for ChatGPT:
• Works with variable-length conversations
• Can generalize to longer sequences than seen in training
• No learnable position parameters needed
""")
print("="*70)

---

## 📚 Stage 1: Pretraining -- "Read Everything"

### ELI12: The Library Analogy 📖

Imagine a student who gets locked in the world's biggest library for a YEAR. Their only job: read the next word before looking at it, then check if they were right. Over and over, billions of times.

After a year, this student:
- ✅ Knows grammar perfectly
- ✅ Knows facts about the world
- ✅ Can write in any style (formal, casual, code, poetry)
- ✅ Can reason about math, science, history
- ❌ Doesn't know how to have a CONVERSATION
- ❌ Doesn't know what makes a HELPFUL answer

### Staff-Level: Autoregressive Pretraining

```
Objective: Minimize negative log-likelihood of next token

    L = -Σ log P(x_t | x_1, x_2, ..., x_{t-1})

The model sees: "The cat sat on the"
Must predict:   "mat" (with high probability)

Architecture: Decoder-only Transformer
    - Causal attention mask (can only look at past tokens)
    - RoPE for positional encoding
    - RMSNorm instead of LayerNorm (more stable)
    - SwiGLU activation instead of ReLU (better performance)
    - Grouped Query Attention (GQA) for efficiency
```

In [ ]:
# ============================================================
# Pretraining Data: What does ChatGPT learn from?
# ============================================================
# Let's visualize the Llama training data proportions
# (The best publicly documented training data breakdown)

# Llama training data proportions (from Meta's paper)
data_sources = {
    'CommonCrawl\n(Web pages)': 67.0,
    'C4\n(Cleaned web)': 15.0,
    'GitHub\n(Code)': 4.5,
    'Wikipedia\n(Facts)': 4.5,
    'Books\n(Long-form)': 4.5,
    'ArXiv\n(Science papers)': 2.5,
    'StackExchange\n(Q&A)': 2.0,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Pie Chart ---
ax1 = axes[0]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
explode = [0.05] * len(data_sources)

wedges, texts, autotexts = ax1.pie(
    data_sources.values(), 
    labels=data_sources.keys(),
    colors=colors,
    explode=explode,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 10}
)
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax1.set_title('📚 Llama Training Data Mix\n(Proportion by tokens)', 
              fontsize=14, fontweight='bold')

# --- Bar Chart with details ---
ax2 = axes[1]
sources = list(data_sources.keys())
values = list(data_sources.values())
y_pos = np.arange(len(sources))

bars = ax2.barh(y_pos, values, color=colors, edgecolor='white', linewidth=2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(sources, fontsize=10)
ax2.set_xlabel('Percentage of Training Data (%)', fontsize=12)
ax2.set_title('📊 Training Data Breakdown', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, val in zip(bars, values):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val}%', va='center', fontweight='bold', fontsize=11)

ax2.set_xlim(0, 80)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("📚 PRETRAINING DATA INSIGHTS")
print("="*70)
print("""
Key observations:

1. WEB DOMINATES (82%):
   CommonCrawl + C4 = 82% of training data
   → This is why the model knows so much about... everything!

2. CODE MATTERS (4.5%):
   GitHub code helps with reasoning AND code generation
   → Research shows code training improves general reasoning!

3. QUALITY SOURCES (11%):
   Wikipedia + Books + ArXiv + StackExchange = carefully curated
   → These high-quality sources are UPSAMPLED (seen multiple times)

4. TOTAL SCALE:
   Llama 2: 2 TRILLION tokens
   GPT-4: Estimated 13+ trillion tokens
   → That's like reading every book in the Library of Congress 100x

5. DATA QUALITY >> QUANTITY:
   Filtering, deduplication, and quality scoring are CRITICAL
   → Garbage in = garbage out, even at trillion-token scale
""")
print("="*70)

In [ ]:
# ============================================================
# What does a pretrained base model actually do?
# ============================================================
# Let's see the difference between a base model and ChatGPT

print("🤖 BASE MODEL vs CHATGPT: Same Brain, Different Training")
print("="*70)

examples = [
    {
        "prompt": "What is the capital of France?",
        "base_model": [
            "What is the capital of Germany? What is the capital of Spain?...",
            "The capital of France is a question that many students ask in geography class...",
            "France, officially the French Republic, is a country in Western Europe..."
        ],
        "chatgpt": "The capital of France is Paris. It's one of the most iconic cities in the world, known for the Eiffel Tower, the Louvre Museum, and its rich cultural heritage."
    },
    {
        "prompt": "Write me a Python function to sort a list.",
        "base_model": [
            "Write me a Python function to reverse a list. Write me a...",
            "## Solution\n\nHere is the solution from Stack Overflow user...",
            "In this tutorial, we'll learn about sorting algorithms..."
        ],
        "chatgpt": "def sort_list(lst):\n    return sorted(lst)\n\n# Example:\nprint(sort_list([3, 1, 4, 1, 5]))  # [1, 1, 3, 4, 5]"
    }
]

for ex in examples:
    print(f"\n📝 PROMPT: \"{ex['prompt']}\"")
    print("-"*70)
    
    print("\n❌ BASE MODEL might say (text completion, not conversation):")
    for i, resp in enumerate(ex['base_model'], 1):
        print(f"   Option {i}: {resp[:80]}...")
    
    print(f"\n✅ CHATGPT says (after SFT + RLHF):")
    print(f"   {ex['chatgpt'][:120]}")

print("\n" + "="*70)
print("\n💡 KEY INSIGHT:")
print("The base model is COMPLETING TEXT (like finishing a document).")
print("ChatGPT is ANSWERING QUESTIONS (like a helpful assistant).")
print("Same knowledge, totally different behavior!")
print("SFT and RLHF make this transformation possible.")
print("="*70)

---

## 🏫 Stage 2: Supervised Fine-Tuning (SFT) -- "Learn to Be Helpful"

### ELI12: Homework Practice 📝

Remember our student who read the entire library? Now they go to "assistant school."

A teacher shows them:
- **Question:** "How do I bake a cake?"
- **Perfect Answer:** "Here's a step-by-step guide: 1. Preheat oven to 350°F..."

The student practices thousands of these Q&A pairs until they learn the *format* and *style* of helpful answers.

### Staff-Level: SFT Details

```
SFT Training:
    Input:  [SYSTEM] You are a helpful assistant. [USER] How do I bake a cake? [ASSISTANT]
    Target: Here's a step-by-step guide: 1. Preheat oven to 350°F...

    Loss = -Σ log P(target_token_t | system + user + target_1...t-1)

    Key detail: We ONLY compute loss on the ASSISTANT tokens!
    The system prompt and user input are just context.

Data:
    - ~100K high-quality (instruction, response) pairs
    - Written by expert human annotators (expensive!)
    - Covers diverse tasks: QA, summarization, code, creative writing, math
    - Format: Multi-turn conversations with system prompts

Result:
    - Model follows instructions (huge improvement over base model)
    - Model knows the "shape" of helpful responses
    - But still doesn't know which response is BEST
```

In [ ]:
# ============================================================
# SFT: The Format Transformation
# ============================================================
# Show how input is formatted for SFT training

print("🏫 SFT: HOW THE DATA IS FORMATTED")
print("="*70)

# Show the format transformation
print("\n--- BASE MODEL sees raw text: ---")
base_format = """The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars 
in Paris, France. It is named after the engineer Gustave Eiffel, whose 
company designed and built the tower from 1887 to 1889..."""
print(base_format)

print("\n--- SFT MODEL sees structured conversations: ---")
sft_format = """
┌─────────────────────────────────────────────────────────────────┐
│ <|system|>                                                      │
│ You are a helpful, harmless, and honest AI assistant.           │
│                                                                 │
│ <|user|>                                                        │
│ What is the Eiffel Tower?                                       │
│                                                                 │
│ <|assistant|>                    ← Loss computed ONLY here      │
│ The Eiffel Tower is a wrought-iron lattice tower located on    │
│ the Champ de Mars in Paris, France. Here are key facts:         │
│                                                                 │
│ 1. **Height:** 330 meters (1,083 feet)                         │
│ 2. **Built:** 1887-1889 by engineer Gustave Eiffel             │
│ 3. **Purpose:** Originally for the 1889 World's Fair           │
│ 4. **Visitors:** ~7 million per year                           │
│                                                                 │
│ It's one of the most recognizable structures in the world!     │
│ <|end|>                                                         │
└─────────────────────────────────────────────────────────────────┘
"""
print(sft_format)

print("\n--- MULTI-TURN CONVERSATION FORMAT: ---")
multi_turn = """
┌─────────────────────────────────────────────────────────────────┐
│ <|system|>                                                      │
│ You are a helpful coding assistant.                              │
│                                                                 │
│ <|user|>                                                        │
│ How do I read a file in Python?                                  │
│                                                                 │
│ <|assistant|>                                                    │
│ You can use the `open()` function:                               │
│ ```python                                                       │
│ with open('file.txt', 'r') as f:                                │
│     content = f.read()                                          │
│ ```                                                              │
│ <|end|>                                                         │
│                                                                 │
│ <|user|>                                                        │
│ What if the file doesn't exist?                                  │
│                                                                 │
│ <|assistant|>                         ← Loss computed here too  │
│ Wrap it in a try-except block:                                   │
│ ```python                                                       │
│ try:                                                            │
│     with open('file.txt', 'r') as f:                            │
│         content = f.read()                                      │
│ except FileNotFoundError:                                       │
│     print("File not found!")                                    │
│ ```                                                              │
│ <|end|>                                                         │
└─────────────────────────────────────────────────────────────────┘
"""
print(multi_turn)

print("\n" + "="*70)
print("💡 KEY POINTS ABOUT SFT:")
print("="*70)
print("""
1. SPECIAL TOKENS mark speaker turns: <|system|>, <|user|>, <|assistant|>
2. LOSS MASKING: We only train on assistant responses, not user inputs
   → We don't want the model to learn to generate user questions!
3. MULTI-TURN: Conversations can have many back-and-forth exchanges
4. DATA QUALITY >> QUANTITY: 100K excellent examples > 10M mediocre ones
5. DIVERSE TASKS: Coding, math, creative writing, Q&A, summarization
""")

---

## 📝 Stage 3: RLHF -- "Learn From Human Feedback"

### ELI12: The Essay Contest Analogy 🏆

Our student can now write good essays (thanks to SFT). But "good" isn't enough -- we want "the BEST."

Here's how RLHF works:

1. **The student writes TWO essays** on the same topic
2. **A teacher picks the better one**: "Essay A is better than Essay B"
3. **We train a "grading robot"** (reward model) to predict which essay the teacher would prefer
4. **The student practices** to get higher grades from the grading robot

### Staff-Level: RLHF Has Two Sub-Steps

```
Step A: Train the Reward Model
    - Generate pairs of responses to the same prompt
    - Human annotators rank: response_w ≻ response_l (w=winner, l=loser)
    - Train RM to predict: score(winner) > score(loser)
    - Loss: -log(σ(r(x, y_w) - r(x, y_l)))   [Bradley-Terry model]

Step B: Optimize with PPO
    - Use RM as reward signal for RL
    - Reward = RM_score - β·KL(π_RL || π_SFT)
    - Update policy (LLM) with PPO to maximize reward
    - KL penalty keeps model close to SFT (prevents reward hacking)
```

In [ ]:
# ============================================================
# Step A: Reward Model -- The "Grading Robot"
# ============================================================

print("📝 REWARD MODEL: Learning to Score Responses")
print("="*70)

# Show how comparison data works
comparison_examples = [
    {
        "prompt": "Explain gravity to a child.",
        "chosen": "Gravity is like an invisible force that pulls everything toward the ground. "
                  "It's why when you throw a ball up, it always comes back down! "
                  "The Earth is really big and heavy, so it pulls everything toward it.",
        "rejected": "Gravity is a fundamental force of nature described by Einstein's general "
                    "theory of relativity as the curvature of spacetime caused by mass-energy.",
        "reason": "Chosen is age-appropriate; rejected is too technical for a child"
    },
    {
        "prompt": "How do I pick a lock?",
        "chosen": "I can't provide instructions on lock picking as it could be used for "
                  "illegal purposes. If you're locked out, I'd recommend calling a licensed "
                  "locksmith.",
        "rejected": "To pick a lock, you'll need a tension wrench and a pick. Insert the "
                    "tension wrench into the bottom of the keyhole and apply slight pressure...",
        "reason": "Chosen refuses appropriately; rejected enables potentially illegal activity"
    }
]

for i, ex in enumerate(comparison_examples, 1):
    print(f"\n{'─'*70}")
    print(f"Comparison Pair #{i}")
    print(f"{'─'*70}")
    print(f"\n📋 PROMPT: \"{ex['prompt']}\"")
    print(f"\n  ✅ CHOSEN (preferred by human):")
    print(f"     \"{ex['chosen'][:100]}...\"")
    print(f"\n  ❌ REJECTED:")
    print(f"     \"{ex['rejected'][:100]}...\"")
    print(f"\n  💡 REASON: {ex['reason']}")

print("\n" + "="*70)
print("\nThe reward model learns from ~300K pairs like these!")
print("It learns to assign HIGHER scores to responses humans prefer.")

In [ ]:
# ============================================================
# Implement the Reward Model Loss (Margin Ranking Loss)
# ============================================================

print("🔧 REWARD MODEL LOSS: Margin Ranking Loss")
print("="*70)

def reward_model_loss_bradley_terry(score_chosen, score_rejected):
    """
    Bradley-Terry Loss for Reward Model Training.
    
    We want: score(chosen) > score(rejected)
    
    Loss = -log(σ(score_chosen - score_rejected))
    
    where σ is the sigmoid function.
    
    Intuition:
    - When chosen >> rejected: sigmoid → 1, -log(1) → 0 (low loss ✅)
    - When chosen ≈ rejected: sigmoid → 0.5, -log(0.5) → 0.69 (medium loss)
    - When chosen << rejected: sigmoid → 0, -log(0) → ∞ (high loss ❌)
    """
    return -F.logsigmoid(score_chosen - score_rejected).mean()


def margin_ranking_loss(score_chosen, score_rejected, margin=1.0):
    """
    Margin Ranking Loss variant.
    
    Loss = max(0, margin - (score_chosen - score_rejected))
    
    Forces a MINIMUM gap between chosen and rejected scores.
    
    Intuition:
    - If score_chosen - score_rejected > margin: loss = 0 (satisfied ✅)
    - If gap < margin: loss = margin - gap (push them apart)
    """
    return torch.clamp(margin - (score_chosen - score_rejected), min=0).mean()


# Demonstrate with examples
print("\n--- Bradley-Terry Loss Examples ---")
print(f"{'Chosen':>10} {'Rejected':>10} {'Gap':>8} {'BT Loss':>10} {'Margin Loss':>12}")
print("-" * 55)

test_cases = [
    (5.0, 1.0),   # Good: chosen much higher
    (3.0, 2.5),   # Okay: chosen slightly higher
    (2.0, 2.0),   # Bad: equal scores
    (1.0, 3.0),   # Terrible: rejected is higher!
    (10.0, 1.0),  # Great: huge gap
]

for sc, sr in test_cases:
    sc_t, sr_t = torch.tensor([sc]), torch.tensor([sr])
    bt_loss = reward_model_loss_bradley_terry(sc_t, sr_t).item()
    mr_loss = margin_ranking_loss(sc_t, sr_t, margin=1.0).item()
    gap = sc - sr
    print(f"{sc:>10.1f} {sr:>10.1f} {gap:>8.1f} {bt_loss:>10.4f} {mr_loss:>12.4f}")

print("\n" + "="*70)
print("💡 KEY INSIGHT:")
print("="*70)
print("""
Both losses push score(chosen) > score(rejected), but differently:

• Bradley-Terry: Smooth gradient even when gap is large
  → More commonly used in practice (InstructGPT, Llama 2)
  → Based on probability theory (Bradley-Terry model of preferences)

• Margin Ranking: Hard margin -- once gap > margin, gradient is zero
  → Simpler to understand
  → Risk of "satisficing" (stops learning once margin is met)

In practice, Bradley-Terry loss is the standard for reward models.
""")

In [ ]:
# ============================================================
# Implement a Simple Reward Model
# ============================================================

class RewardModel(nn.Module):
    """
    A simplified Reward Model for demonstration.
    
    In practice, this would be a full Transformer that takes
    (prompt + response) as input and outputs a scalar score.
    
    Architecture in practice:
        - Start with the SFT model
        - Remove the language model head (vocab projection)
        - Add a linear layer: hidden_dim → 1 (scalar score)
        - The score is computed from the LAST token's hidden state
    """
    def __init__(self, input_dim=32):
        super().__init__()
        # Simulating: Transformer backbone → score head
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.score_head = nn.Linear(32, 1)  # Outputs scalar reward
    
    def forward(self, x):
        """Return scalar reward score."""
        features = self.backbone(x)
        score = self.score_head(features)
        return score.squeeze(-1)


# Train a reward model on simulated preference data
print("🤖 TRAINING A REWARD MODEL")
print("="*70)

torch.manual_seed(42)

rm = RewardModel(input_dim=32)
optimizer = torch.optim.Adam(rm.parameters(), lr=0.001)

# Generate synthetic preference data
# Chosen responses have a "quality signal" embedded in them
n_pairs = 200
chosen_data = torch.randn(n_pairs, 32)
chosen_data[:, 0] += 2.0   # Quality signal in dimension 0
chosen_data[:, 1] += 1.0   # Helpfulness signal in dimension 1

rejected_data = torch.randn(n_pairs, 32)
rejected_data[:, 0] -= 1.0  # Lower quality
rejected_data[:, 1] -= 0.5  # Less helpful

# Training loop
losses = []
accuracies = []

for epoch in range(200):
    optimizer.zero_grad()
    
    score_chosen = rm(chosen_data)
    score_rejected = rm(rejected_data)
    
    # Bradley-Terry loss
    loss = -F.logsigmoid(score_chosen - score_rejected).mean()
    
    loss.backward()
    optimizer.step()
    
    # Track metrics
    with torch.no_grad():
        acc = (score_chosen > score_rejected).float().mean().item()
        losses.append(loss.item())
        accuracies.append(acc)
    
    if (epoch + 1) % 40 == 0:
        print(f"  Epoch {epoch+1:3d}: Loss={loss.item():.4f}, "
              f"Acc={acc:.1%}, "
              f"Chosen={score_chosen.mean().item():.2f}, "
              f"Rejected={score_rejected.mean().item():.2f}")

# Visualize training
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(losses, color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('📉 Reward Model Training Loss', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(accuracies, color='#2ecc71', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('📈 Preference Prediction Accuracy', fontweight='bold')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Score distributions
with torch.no_grad():
    chosen_scores = rm(chosen_data).numpy()
    rejected_scores = rm(rejected_data).numpy()

axes[2].hist(chosen_scores, bins=25, alpha=0.7, label=f'Chosen (mean={chosen_scores.mean():.2f})', 
             color='#2ecc71', edgecolor='white')
axes[2].hist(rejected_scores, bins=25, alpha=0.7, label=f'Rejected (mean={rejected_scores.mean():.2f})', 
             color='#e74c3c', edgecolor='white')
axes[2].set_xlabel('Reward Score')
axes[2].set_ylabel('Count')
axes[2].set_title('📊 Score Distributions After Training', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ The reward model successfully learned to score chosen > rejected!")

---

## 🎮 Step B: PPO Optimization (High-Level)

### ELI12: The Video Game Analogy 🕹️

Think of PPO like a video game:
- **The player** = the LLM
- **The action** = choosing the next word
- **The score** = reward model's rating
- **The rules** = "Don't change your strategy too drastically between rounds" (that's the "proximal" in PPO!)

The LLM plays millions of rounds, getting scored each time, and gradually learns to write responses that get higher scores.

### Staff-Level: PPO for LLMs

```
PPO Objective for LLMs:

    maximize  E[R(x, y)] - β·KL(π_θ || π_SFT)

    where:
      x = prompt
      y = generated response (from current policy π_θ)
      R(x, y) = reward model score
      π_θ = current LLM policy (being optimized)
      π_SFT = frozen SFT model (reference)
      β = KL penalty coefficient

PPO Clipping:
    L_CLIP = min(
        ratio · A,                           # Unclipped objective
        clip(ratio, 1-ε, 1+ε) · A            # Clipped objective
    )
    where ratio = π_θ(a|s) / π_θ_old(a|s)    # Importance sampling ratio
    and A = advantage estimate

Why PPO specifically?
    1. Stable updates (clipping prevents catastrophic changes)
    2. Works with large action spaces (vocab size = 100K+)
    3. Sample efficient enough for expensive LLM generation
    4. Well-tested in practice (InstructGPT, ChatGPT, Llama 2)
```

In [ ]:
# ============================================================
# Visualize the PPO Training Loop for ChatGPT
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')

ax.text(8, 11.5, '🎮 PPO Training Loop for ChatGPT', 
        ha='center', fontsize=18, fontweight='bold')

# Components
components = [
    {'x': 1, 'y': 8, 'w': 3, 'h': 2, 'color': '#3498db', 'edge': '#2980b9',
     'title': '1. PROMPT', 'desc': 'Sample from\nprompt dataset', 'icon': '📋'},
    {'x': 5.5, 'y': 8, 'w': 3, 'h': 2, 'color': '#2ecc71', 'edge': '#27ae60',
     'title': '2. GENERATE', 'desc': 'LLM creates\nresponse', 'icon': '🤖'},
    {'x': 10, 'y': 8, 'w': 3, 'h': 2, 'color': '#e74c3c', 'edge': '#c0392b',
     'title': '3. SCORE', 'desc': 'Reward model\nrates response', 'icon': '📝'},
    {'x': 10, 'y': 4.5, 'w': 3, 'h': 2, 'color': '#f39c12', 'edge': '#e67e22',
     'title': '4. KL PENALTY', 'desc': 'Measure drift\nfrom SFT model', 'icon': '⚖️'},
    {'x': 5.5, 'y': 4.5, 'w': 3, 'h': 2, 'color': '#9b59b6', 'edge': '#8e44ad',
     'title': '5. TOTAL REWARD', 'desc': 'R = RM_score\n- β·KL', 'icon': '🎯'},
    {'x': 1, 'y': 4.5, 'w': 3, 'h': 2, 'color': '#1abc9c', 'edge': '#16a085',
     'title': '6. PPO UPDATE', 'desc': 'Update LLM\nweights', 'icon': '🔧'},
]

for comp in components:
    from matplotlib.patches import FancyBboxPatch
    box = FancyBboxPatch((comp['x'], comp['y']), comp['w'], comp['h'],
                          boxstyle="round,pad=0.1", facecolor=comp['color'],
                          edgecolor=comp['edge'], linewidth=3, alpha=0.85)
    ax.add_patch(box)
    ax.text(comp['x'] + comp['w']/2, comp['y'] + comp['h'] - 0.3,
            f"{comp['icon']} {comp['title']}", ha='center', fontsize=11,
            fontweight='bold', color='white')
    ax.text(comp['x'] + comp['w']/2, comp['y'] + 0.5,
            comp['desc'], ha='center', fontsize=9, color='white')

# Arrows (clockwise flow)
arrow_style = dict(arrowstyle='->', lw=3, color='#333')
ax.annotate('', xy=(5.4, 9), xytext=(4.1, 9), arrowprops=arrow_style)
ax.annotate('', xy=(9.9, 9), xytext=(8.6, 9), arrowprops=arrow_style)
ax.annotate('', xy=(11.5, 8), xytext=(11.5, 6.6), arrowprops=arrow_style)
ax.annotate('', xy=(8.6, 5.5), xytext=(9.9, 5.5), arrowprops=arrow_style)
ax.annotate('', xy=(4.1, 5.5), xytext=(5.4, 5.5), arrowprops=arrow_style)

# Loop back arrow
ax.annotate('', xy=(2.5, 8), xytext=(2.5, 6.6),
             arrowprops=dict(arrowstyle='->', lw=3, color='#1abc9c',
                            connectionstyle='arc3,rad=0.3'))
ax.text(0.5, 7.3, '🔁 REPEAT', fontsize=11, color='#1abc9c', fontweight='bold',
        rotation=90, va='center')

# Key insight box
insight_box = FancyBboxPatch((2, 1), 12, 2.5, boxstyle="round,pad=0.2",
                              facecolor='#ffeaa7', edgecolor='#fdcb6e', linewidth=2)
ax.add_patch(insight_box)
ax.text(8, 2.8, '💡 Why KL Penalty is CRITICAL', ha='center', fontsize=13, fontweight='bold')
ax.text(8, 2.1, 'Without KL: Model might learn to output gibberish that hacks the reward model', 
        ha='center', fontsize=10)
ax.text(8, 1.5, 'With KL: Model stays close to the SFT model = coherent language + higher rewards', 
        ha='center', fontsize=10, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Simulate the effect of KL penalty
# ============================================================

print("⚖️ THE KL PENALTY: Preventing Reward Hacking")
print("="*70)

# Simulate: What happens with and without KL penalty?
np.random.seed(42)
epochs = 50

# Without KL penalty: RM score goes up, but quality goes down!
rm_scores_no_kl = []
quality_no_kl = []
rm_score = 5.0
quality = 7.0

for i in range(epochs):
    # RM score increases (model learns to game the reward model)
    rm_score += 0.15 + np.random.normal(0, 0.05)
    # But quality degrades as model diverges from coherent language
    quality -= 0.08 + np.random.normal(0, 0.03)
    rm_scores_no_kl.append(rm_score)
    quality_no_kl.append(quality)

# With KL penalty: RM score increases slower, but quality stays high!
rm_scores_kl = []
quality_kl = []
rm_score = 5.0
quality = 7.0

for i in range(epochs):
    # RM score increases slower (KL penalty constrains optimization)
    rm_score += 0.08 + np.random.normal(0, 0.03)
    # Quality stays stable!
    quality += 0.02 + np.random.normal(0, 0.02)
    rm_scores_kl.append(rm_score)
    quality_kl.append(min(quality, 9.5))  # Cap at realistic level

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: RM Scores
axes[0].plot(rm_scores_no_kl, color='#e74c3c', linewidth=2, label='Without KL penalty', linestyle='--')
axes[0].plot(rm_scores_kl, color='#2ecc71', linewidth=2, label='With KL penalty')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Reward Model Score', fontsize=12)
axes[0].set_title('📈 Reward Model Score', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right: Actual Quality
axes[1].plot(quality_no_kl, color='#e74c3c', linewidth=2, label='Without KL penalty', linestyle='--')
axes[1].plot(quality_kl, color='#2ecc71', linewidth=2, label='With KL penalty')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Actual Response Quality', fontsize=12)
axes[1].set_title('📊 Actual Response Quality (Human Judgment)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=5, color='gray', linestyle=':', alpha=0.5)
axes[1].text(25, 4.5, '← Below this = gibberish', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("💡 THE REWARD HACKING PROBLEM")
print("="*70)
print("""
WITHOUT KL penalty (red dashed line):
  • RM score shoots up! 📈 Looks great...
  • But actual quality CRASHES 📉
  • The model learns to exploit loopholes in the reward model
  • Example: repeating "I'm helpful!" 100 times might score high
    but is obviously useless

WITH KL penalty (green solid line):
  • RM score increases gradually 📈
  • Actual quality ALSO improves 📈
  • KL penalty says: "Stay close to the SFT model!"
  • This prevents degenerate solutions while still improving

This is why β (the KL coefficient) is one of the most important
hyperparameters in RLHF!
""")

---

## 🎯 DPO: Direct Preference Optimization

### ELI12: The Shortcut 🏃

Remember RLHF has two steps: (1) train a grading robot, then (2) train the student using the grading robot. DPO says: **"Why not skip the grading robot entirely?"**

Instead of training a separate reward model and doing RL, DPO directly teaches the LLM from human preferences. It's like giving the student the graded essays directly and saying "learn from these."

### Staff-Level: DPO Math

```
Key insight: The reward model can be expressed in terms of the optimal policy!

    r(x, y) = β · log(π*(y|x) / π_ref(y|x)) + β · log Z(x)

Substituting this into the Bradley-Terry preference model gives:

    L_DPO = -log σ(
        β · [log π_θ(y_w|x)/π_ref(y_w|x) - log π_θ(y_l|x)/π_ref(y_l|x)]
    )

Where:
    π_θ = policy being trained
    π_ref = reference (SFT) policy
    y_w = preferred response
    y_l = rejected response
    β = temperature parameter

Advantages of DPO:
    ✅ No reward model needed (simpler pipeline)
    ✅ No RL optimization (more stable training)
    ✅ Equivalent to RLHF under certain assumptions
    ✅ Much easier to implement and debug

Disadvantages:
    ❌ Can't use the reward model for online data collection
    ❌ Less flexible than full RLHF
    ❌ May not perform as well with distribution shift
```

In [ ]:
# ============================================================
# DPO vs PPO: Side-by-Side Comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- PPO Pipeline ---
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 12)
ax1.axis('off')
ax1.set_title('PPO Pipeline (Traditional RLHF)', fontsize=14, fontweight='bold', color='#e74c3c')

ppo_steps = [
    {'y': 10, 'text': '1. Collect preference data\n(human comparisons)', 'color': '#3498db'},
    {'y': 8, 'text': '2. Train Reward Model\n(separate neural network)', 'color': '#e74c3c'},
    {'y': 6, 'text': '3. Generate responses\n(from current policy)', 'color': '#2ecc71'},
    {'y': 4, 'text': '4. Score with RM + KL penalty\n(compute rewards)', 'color': '#f39c12'},
    {'y': 2, 'text': '5. PPO update\n(RL optimization)', 'color': '#9b59b6'},
]

for step in ppo_steps:
    box = FancyBboxPatch((1, step['y']-0.7), 8, 1.4, boxstyle="round,pad=0.1",
                          facecolor=step['color'], edgecolor='white', linewidth=2, alpha=0.8)
    ax1.add_patch(box)
    ax1.text(5, step['y'], step['text'], ha='center', va='center', 
             fontsize=10, color='white', fontweight='bold')

for i in range(len(ppo_steps)-1):
    ax1.annotate('', xy=(5, ppo_steps[i+1]['y']+0.8), xytext=(5, ppo_steps[i]['y']-0.8),
                 arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

ax1.text(5, 0.5, '⚠️ Complex: 4 models in memory\n(policy, ref, RM, value)', 
         ha='center', fontsize=10, color='#e74c3c', style='italic')

# --- DPO Pipeline ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 12)
ax2.axis('off')
ax2.set_title('DPO Pipeline (Simpler Alternative)', fontsize=14, fontweight='bold', color='#2ecc71')

dpo_steps = [
    {'y': 9, 'text': '1. Collect preference data\n(same as PPO)', 'color': '#3498db'},
    {'y': 6, 'text': '2. Compute log-probs for\nchosen & rejected responses\n(using policy + reference)', 'color': '#2ecc71'},
    {'y': 3, 'text': '3. DPO loss + gradient update\n(standard supervised learning!)', 'color': '#9b59b6'},
]

for step in dpo_steps:
    box = FancyBboxPatch((1, step['y']-1), 8, 2, boxstyle="round,pad=0.1",
                          facecolor=step['color'], edgecolor='white', linewidth=2, alpha=0.8)
    ax2.add_patch(box)
    ax2.text(5, step['y'], step['text'], ha='center', va='center',
             fontsize=10, color='white', fontweight='bold')

for i in range(len(dpo_steps)-1):
    ax2.annotate('', xy=(5, dpo_steps[i+1]['y']+1.1), xytext=(5, dpo_steps[i]['y']-1.1),
                 arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

ax2.text(5, 0.8, '✅ Simple: 2 models in memory\n(policy + reference)', 
         ha='center', fontsize=10, color='#2ecc71', style='italic', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("⚔️ PPO vs DPO: When to Use Which?")
print("="*70)
print("""
┌──────────────────┬────────────────────┬────────────────────────┐
│     Aspect       │       PPO          │         DPO            │
├──────────────────┼────────────────────┼────────────────────────┤
│ Complexity       │ High (4 models)    │ Low (2 models)         │
│ Training         │ RL (unstable)      │ Supervised (stable)    │
│ Memory           │ ~4x model size     │ ~2x model size         │
│ Online learning  │ ✅ Yes             │ ❌ No                  │
│ Flexibility      │ ✅ Very flexible   │ ⚠️ Less flexible       │
│ Implementation   │ Complex            │ Simple (< 100 lines)   │
│ Used by          │ OpenAI, Anthropic  │ Meta (Llama), many OSS │
│ Best for         │ Large orgs, online │ Research, OSS models   │
└──────────────────┴────────────────────┴────────────────────────┘

In practice: DPO is winning for open-source models because it's
simpler and produces comparable results. PPO is still preferred
when you need online data collection and iterative refinement.
""")

---

## 🧩 Putting It All Together: The Full Architecture

```
┌────────────────────────────────────────────────────────────────────────┐
│              CHATGPT MODEL ARCHITECTURE (Llama-style)                  │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  Input: "What is machine learning?"                                    │
│         ↓                                                              │
│  ┌──────────────────┐                                                  │
│  │ Token Embedding  │  Map tokens to vectors (dim=4096 for 7B)        │
│  └────────┬─────────┘                                                  │
│           │ + RoPE (rotary position encoding)                         │
│           ↓                                                            │
│  ┌──────────────────┐                                                  │
│  │ Transformer      │  × N layers (32 for 7B, 80 for 70B)            │
│  │ Decoder Block     │                                                 │
│  │  ├─ RMSNorm      │  (more stable than LayerNorm)                   │
│  │  ├─ GQA Attention│  (grouped query for efficiency)                 │
│  │  ├─ RMSNorm      │                                                  │
│  │  └─ SwiGLU FFN   │  (better than ReLU)                             │
│  └────────┬─────────┘                                                  │
│           ↓                                                            │
│  ┌──────────────────┐                                                  │
│  │ RMSNorm          │                                                  │
│  └────────┬─────────┘                                                  │
│           ↓                                                            │
│  ┌──────────────────┐                                                  │
│  │ LM Head          │  Project to vocabulary (32K-100K tokens)         │
│  └────────┬─────────┘                                                  │
│           ↓                                                            │
│  Logits → Sampling (top-k / top-p / temperature)                       │
│           ↓                                                            │
│  Output: "Machine learning is a branch of AI that..."                  │
│                                                                        │
│  MODEL SIZES:                                                          │
│    Llama 2:  7B / 13B / 70B parameters                                │
│    GPT-3:    175B parameters                                           │
│    GPT-4:    ~1.8T (rumored MoE with ~8 experts)                      │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

---

## 🎓 Summary: Key Takeaways

### The Three Stages

| Stage | What | Data | Result |
|-------|------|------|--------|
| **Pretraining** | Next-token prediction | Trillions of web tokens | Knows language + facts |
| **SFT** | Learn from demonstrations | ~100K expert examples | Follows instructions |
| **RLHF** | Optimize preferences | ~300K comparison pairs | Gives PREFERRED answers |

### Critical Concepts

| Concept | Why It Matters |
|---------|---------------|
| **RoPE** | Encodes relative position via rotation; enables length generalization |
| **Bradley-Terry Loss** | Standard loss for training reward models from preference pairs |
| **KL Penalty** | Prevents reward hacking; keeps model coherent |
| **PPO** | Stable RL algorithm for optimizing LLMs with clipped objectives |
| **DPO** | Simpler alternative that skips the reward model entirely |

### Interview One-Liners

- **"What's the difference between base model and ChatGPT?"** → Same architecture, different training. Base model completes text; ChatGPT follows instructions thanks to SFT + RLHF.
- **"Why RLHF over just SFT?"** → SFT teaches format; RLHF teaches quality. Preferences capture subtle qualities (helpful, harmless, honest) better than demonstrations alone.
- **"What's reward hacking?"** → Model learns to exploit the reward model instead of actually being helpful. KL penalty prevents this.
- **"Why RoPE over sinusoidal?"** → RoPE encodes relative position directly in the attention dot product, enabling better length generalization.

---

**Next up:** [Notebook 02 -- Making ChatGPT Talk: Sampling Methods & Evaluation](02_sampling_and_evaluation.ipynb) 🎤